## 📦 Environment Setup & Imports

In [1]:
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

load_dotenv(project_root / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

from rag_pipeline import (
    load_data,
    build_or_load_faiss_index,
    check_polypharmacy as check_polypharmacy_orig,
    check_polypharmacy_light as check_polypharmacy_light_orig,
)

kb_df, lookups = load_data()
index, embeddings = build_or_load_faiss_index(kb_df, client)


def check_polypharmacy(drug_query: str):
    text = drug_query.replace("interaction", "").replace("and", ",")
    drugs = [d.strip() for d in text.split(",") if d.strip()]
    return check_polypharmacy_orig(drugs, kb_df, lookups, index, embeddings, client)


def check_polypharmacy_light(drug_query: str):
    text = drug_query.replace("interaction", "").replace("and", ",")
    drugs = [d.strip() for d in text.split(",") if d.strip()]
    return check_polypharmacy_light_orig(drugs, kb_df, lookups)


print("Environment initialized successfully")

Loading knowledge base...
   Loaded 170,782 interactions
   Unique drug pairs: 170,782

Loading RxNorm mappings...
   Name to RxCUI: 157,972
   Brand to Ingredient: 77,518
Loading existing FAISS index...
   Loaded index with 170,782 vectors (3072 dims)
Environment initialized successfully


## 🎯 Severity Classification (GPT-5)

In [2]:
def classify_severity_gpt5(interaction_text):
    """
    Use GPT-5 to classify drug interaction severity.

    Args:
        interaction_text: Single interaction description from KB

    Returns:
        dict with severity (RED/YELLOW/GREEN) and brief clinical reasoning
    """
    prompt = f"""You are a medical doctor and clinical pharmacist reviewing a drug-drug interaction.

Interaction evidence:
{interaction_text}

Task:
Determine the CLINICAL SEVERITY of this interaction even if words such as
"contraindicated", "major", "moderate", or "minor" do NOT appear.
Use both the provided evidence and your own pharmacologic knowledge.

Classification rules:
- 🟥 (Contraindicated): Life-threatening or severe interaction - avoid combination entirely.
- 🟨 (Caution): Clinically significant or moderate risk - requires monitoring or dose adjustment.
- 🟩 (No Interaction): No meaningful pharmacologic or clinical interaction expected.

Return ONLY valid JSON - no markdown, no commentary, no code fences.
Output must start with {{ and end with }}.

Expected format:
{{"severity": "🟥" or "🟨" or "🟩", "explanation": "brief reasoning"}}

Examples:
- "may increase bleeding risk" → 🟨
- "contraindicated with..." → 🟥
- "no interaction known" or "minimal clinical effect" → 🟩
"""
    try:
        response = client.responses.create(
            model="gpt-5",
            input=prompt,
            text={"format": {"type": "text"}}
        )

        result_text = (response.output_text or "").strip()

        if not result_text or not result_text.startswith("{") or not result_text.endswith("}"):
            raise ValueError("Invalid JSON format from GPT-5")

        return json.loads(result_text)

    except (json.JSONDecodeError, Exception) as e:
        raise


print("Severity classifier defined")

Severity classifier defined


In [3]:
# Test the classifier
test_interaction = "The metabolism of Diphenhydramine can be decreased when combined with Acetaminophen."
result = classify_severity_gpt5(test_interaction)
print(f"   Evidence: {test_interaction}")
print(f"   Severity: {result['severity']}")
print(f"   Reasoning: {result['explanation']}")

   Evidence: The metabolism of Diphenhydramine can be decreased when combined with Acetaminophen.
   Severity: 🟩
   Reasoning: Acetaminophen is not a meaningful inhibitor of CYP2D6 (primary pathway for diphenhydramine). Any reduction in diphenhydramine metabolism is minimal and not clinically significant; the combination is widely co-formulated without dose adjustment.


## 🔄 Tier 2 Evidence Synthesis

In [4]:
def synthesize_tier2_evidence(hits, drug1, drug2):
    """
    Synthesize cautious Tier-2 guidance from semantically similar interactions.

    Args:
        hits: List of similar drug interactions from FAISS
        drug1, drug2: Query drug names

    Returns:
        str: Clinical synthesis note
    """
    evidence_text = "\n\n".join([
        f"Similar interaction {i + 1} (confidence: {hit['retrieval_score']:.2f}):\n"
        f"{hit['drug1']} + {hit['drug2']}: {hit['evidence']}"
        for i, hit in enumerate(hits[:3])
    ])

    prompt = f"""You are a medical doctor and clinical pharmacist analyzing SIMILAR drug interactions.

Original query: {drug1} and {drug2}
Evidence from SIMILAR drug combinations:
{evidence_text}

Task:
Write a cautious 2-3 sentence clinical note summarizing potential risks
based on pharmacologic or class similarities.
Use both the provided evidence and your own pharmacologic knowledge to ensure
the synthesis is clinically meaningful.

Rules:
- Use phrases like "may exhibit similar interactions", "class-related concerns", 
  "pharmacologically related compounds suggest...".
- Be explicit that this is inferred, not direct evidence.
- Recommend monitoring any adverse effects mentioned.
- Return ONLY valid JSON: {{"synthesis": "clinical note"}}.
"""

    try:
        response = client.responses.create(
            model="gpt-5",
            input=prompt,
            text={"format": {"type": "text"}}
        )

        result_text = (response.output_text or "").strip()

        if not result_text or not result_text.startswith("{") or not result_text.endswith("}"):
            raise ValueError("Invalid JSON format from GPT-5")

        result = json.loads(result_text)
        return result.get("synthesis", "Unable to synthesize evidence.")

    except (json.JSONDecodeError, Exception):
        raise


print("Tier 2 synthesizer defined")

Tier 2 synthesizer defined


In [5]:
print("\n🧪 Testing Tier-2 Synthesis:")

mock_hits = [
    {
        "drug1": "Acetaminophen",
        "drug2": "Warfarin",
        "evidence": "Acetaminophen may enhance the anticoagulant effect of Warfarin, increasing bleeding risk with prolonged use.",
        "retrieval_score": 0.78
    },
    {
        "drug1": "Ibuprofen",
        "drug2": "Warfarin",
        "evidence": "Warfarin may increase the anticoagulant activity of Ibuprofen, leading to additive bleeding risk.",
        "retrieval_score": 0.72
    }
]

# Tier-2 scenario: no direct data for Aspirin + Acetaminophen,
# but semantically similar evidence from Warfarin + analgesic pairs.
synthesis = synthesize_tier2_evidence(mock_hits, "Aspirin", "Acetaminophen")

print(f"   Query: Aspirin + Acetaminophen")
print(f"   Synthesis: {synthesis}")


🧪 Testing Tier-2 Synthesis:
   Query: Aspirin + Acetaminophen
   Synthesis: Aspirin and acetaminophen may exhibit similar interactions inferred from pharmacologically related compounds: evidence that acetaminophen and NSAIDs (e.g., ibuprofen) increase bleeding risk when combined with warfarin suggests class-related concerns for additive bleeding and GI irritation when an antiplatelet NSAID (aspirin) is used with another analgesic, although acetaminophen itself has minimal antiplatelet effect. This is inferred rather than direct evidence for this pair; additional class-related concerns include renal/GI toxicity with chronic high-dose dual-analgesic use and hepatotoxicity from acetaminophen if doses exceed recommendations or with alcohol. Monitor for melena, hematemesis, easy bruising, abdominal pain, and consider periodic renal function, hemoglobin, and liver enzyme checks with prolonged or high-dose therapy.
